In [2]:
import os
from pathlib import Path
import ast
import numpy as np
import mne
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
event_types = ["fixation", "pso", "saccade", "peak_sac_velocity", "motion_energy", "saccade_curvature"]

mnt_dir_camme = "/Users/carmenamme/Desktop/remote_camme"
mnt_dir_rawdir = "/Users/carmenamme/Desktop/remote_rawdir"

raw_dir = f"{mnt_dir_rawdir}/t1_defaced"
plots_dir = f"{mnt_dir_camme}/AVS-saccade-locking/AVS-saccade-locking/results"

significant_sensors_overall_path = f"{mnt_dir_camme}/AVS-saccade-locking/AVS-saccade-locking/results/significant_sensors_overall_with_explained_variance.csv"
significant_sensors_overall = pd.read_csv(significant_sensors_overall_path)


In [5]:
significant_sensors_overall

,sensor,f_stat,p_val,best_event_based_on_min_sd,best_event_median_sd,subject,explained_variance
0,0,11.966653,8.128966e-08,saccade_curvature,3.290516,as01,0.074662
1,3,3.099656,1.597453e-02,saccade_curvature,5.019604,as01,0.035949
2,5,69.107390,3.122336e-22,saccade_curvature,2.298543,as01,0.034890
3,8,2.696342,3.039590e-02,saccade,3.142441,as01,0.027157
4,9,4.561560,1.528745e-03,saccade_curvature,2.954355,as01,0.025890
...,...,...,...,...,...,...,...
126,64,5.460176,4.844637e-04,motion_energy,2.456141,as05,0.002347
127,66,7.143954,3.580478e-05,motion_energy,2.684332,as05,0.002192
128,68,2.463139,4.616940e-02,saccade_curvature,3.926942,as05,0.001934
129,69,3.092269,1.585095e-02,saccade_curvature,2.623705,as05,0.001840


In [6]:
event_colors_df = pd.read_csv(f"{mnt_dir_camme}/AVS-saccade-locking/AVS-saccade-locking/results/all_subjects/all_event_colors.csv")
EVENT_COLORS = {
        row["event"]: (row["R"]/255, row["G"]/255, row["B"]/255)
    for _, row in event_colors_df.iterrows()
}
EVENT_COLORS

{'saccade': (0.5137254901960784, 0.1450980392156863, 0.3137254901960784),
 'peak_sac_velocity': (0.6666666666666666,
  0.29411764705882354,
  0.3137254901960784),
 'motion_energy': (0.7019607843137254,
  0.34509803921568627,
  0.3215686274509804),
 'saccade_curvature': (0.7529411764705882,
  0.4549019607843137,
  0.36470588235294116),
 'fixation': (0.42745098039215684, 0.5647058823529412, 0.7490196078431373),
 'pso': (0.3686274509803922, 0.3411764705882353, 0.6901960784313725)}

In [ ]:
all_colours_all_subjects = []
all_dipoles_all_subjects = []
all_dipole_pos_all_subjects = []
all_expl_var_all_subjects = []
for counter, subject in enumerate([1, 2, 3, 4, 5]):
    all_colours = []
    all_dipoles_all_events = []
    all_dipole_pos_all_events = []
    all_expl_var_all_events = []
    # subject = 1
    subject = f"as{subject:02d}"

    print("subject: ", subject)
    ica_save_path = os.path.join(Path(plots_dir), subject, "ica")
    dip_save_path = os.path.join(ica_save_path, "ica_dipoles")

    # load trans file
    trans_path = f"{mnt_dir_rawdir}/{subject}/mri/transforms/{subject}-trans.fif"
    trans = mne.read_trans(trans_path)

    significant_sensors_subject = significant_sensors_overall[significant_sensors_overall['subject'] == subject]
    for counter, event in enumerate(event_types):
        # if counter == 3:
        #     break
        significant_sensors_event = significant_sensors_subject[significant_sensors_subject['best_event_based_on_min_sd'] == event]
        print(f"Subject {subject}, Event {event}: {len(significant_sensors_event)} significant ICs, Total Explained Variance: {significant_sensors_event['explained_variance'].sum():.4f}")
        
        if len(significant_sensors_event) == 0:
            
            continue
        
        all_dipoles_this_event, all_dipole_pos_this_event, all_colours_this_events, all_expl_var_this_event = [], [], [], []
        for ic in significant_sensors_event.sensor.values.tolist():
            print(f"  Loading dipole for IC {ic}...")
            dip = mne.read_dipole(os.path.join(dip_save_path, f"ica_scene_dipole_fit_mag_ic_{ic}.dip"))
            print(f"    Original pos (head coord): {dip.pos}")
            # dip.ori[:] = 0
            # dip.amplitude = np.array(significant_sensors_event[significant_sensors_event['sensor'] == ic]['explained_variance'])
            
            dip_pos = dip.to_mri(
                subject,
                trans='fsaverage',
                subjects_dir=f"{mnt_dir_rawdir}/t1_defaced/",
            )
            print(f"    Transformed pos (mri coord): {dip_pos}")
            
            # dip.pos = mne.head_to_mni(
            #     dip.pos,
            #     subject,
            #     mri_head_t=trans,
            #     subjects_dir=f"{mnt_dir_rawdir}/t1_defaced/",
            #     verbose=None,
            # )
            
            # xfm = mne.read_talxfm('fsaverage', subjects_dir=f"{mnt_dir_rawdir}/t1_defaced/")
            # pos_m = dip.pos / 1000.0
            # dip.pos = mne.transforms.apply_trans(xfm['trans'], pos_m) * 1000.0

            all_dipoles_this_event.append(dip)
            all_dipole_pos_this_event.append(dip_pos)
            all_colours_this_events.append(EVENT_COLORS[event])
            all_expl_var_this_event.append(float(significant_sensors_event[significant_sensors_event['sensor'] == ic]['explained_variance'].values[0]))
        
        all_dipoles_all_events.append(all_dipoles_this_event)
        all_dipole_pos_all_events.append(all_dipole_pos_this_event)
        all_colours.append(all_colours_this_events)
        all_expl_var_all_events.append(all_expl_var_this_event)
    
        print(len(all_dipoles_this_event))
    
    all_dipoles_all_subjects.append(all_dipoles_all_events)
    all_dipole_pos_all_subjects.append(all_dipole_pos_all_events)
    all_colours_all_subjects.append(all_colours)
    all_expl_var_all_subjects.append(all_expl_var_all_events)
    


subject:  as01
Subject as01, Event fixation: 0 significant ICs, Total Explained Variance: 0.0000
Subject as01, Event pso: 0 significant ICs, Total Explained Variance: 0.0000
Subject as01, Event saccade: 7 significant ICs, Total Explained Variance: 0.1272
  Loading dipole for IC 8...
1 dipole(s) found
    Original pos (head coord): [[0.00578 0.07489 0.0254 ]]
    Transformed pos (mri coord): [[  7.38688415  44.55581092 -11.69297502]]
  Loading dipole for IC 15...
1 dipole(s) found
    Original pos (head coord): [[ 0.06021 -0.00208  0.06992]]
    Transformed pos (mri coord): [[ 62.0903507  -34.6178804   28.40808645]]
  Loading dipole for IC 16...
1 dipole(s) found
    Original pos (head coord): [[0.02433 0.07188 0.05501]]
    Transformed pos (mri coord): [[25.94746069 39.93983534 17.70281606]]
  Loading dipole for IC 19...
1 dipole(s) found
    Original pos (head coord): [[-0.06427  0.01201  0.01663]]
    Transformed pos (mri coord): [[-62.43897402 -17.97508745 -24.02369404]]
  Loading d

In [8]:
# unnest colours and dipoles (they are nested by subject and event)
all_colours = [colour for subject_colours in all_colours_all_subjects for event_colours in subject_colours for colour in event_colours]
all_dipoles = [dipole for subject_dipoles in all_dipoles_all_subjects for event_dipoles in subject_dipoles for dipole in event_dipoles]
all_expl_var = [expl_var for subject_expl_var in all_expl_var_all_subjects for event_expl_var in subject_expl_var for expl_var in event_expl_var]
all_dipole_pos = [dipole_pos for subject_dipole_pos in all_dipole_pos_all_subjects for event_dipole_pos in subject_dipole_pos for dipole_pos in event_dipole_pos]
len(all_dipoles)

131

In [9]:
brain_kwargs = dict(alpha=0.1, background="white", cortex="low_contrast")
brain = mne.viz.Brain(
    'fsaverage',
    hemi='both',
    # surf='inflated',
    subjects_dir=mnt_dir_rawdir,
    **brain_kwargs,
)

Using pyvistaqt 3d backend.


In [ ]:
# for counter, dip in enumerate(all_dipoles):
    # brain.add_dipole()
    #     dip,
    #     trans='fsaverage',
    #     colors=[all_colours[counter]],
    #     alpha=1,
    #     mode="arrow",
    #     scales=all_expl_var[counter]*300,
    #     )
#     print(dip.pos)

for counter, dip_loc in enumerate(all_dipole_pos):
    brain.add_foci(
        dip_loc,
        coords_as_verts=False,
        # map_surface='fsaverage',
        hemi='vol',
        color=all_colours[counter],
        alpha=0.8,
        scale_factor=all_expl_var[counter]*20,
    )


In [45]:
all_dipole_pos

[array([[  7.38688415,  44.55581092, -11.69297502]])]

In [29]:
np.array([dip.pos for dip in all_dipoles]).reshape(-1, 3)[0]

array([0.00578, 0.07489, 0.0254 ])

In [38]:
all_colours[counter]


(0.5137254901960784, 0.1450980392156863, 0.3137254901960784)